In [1]:
import os

In [2]:
%pwd

'c:\\projects\\SellWise\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\projects\\SellWise'

In [6]:
from dotenv import load_dotenv

# This will search for the .env file in your directory and load its variables
load_dotenv()

True

In [7]:
# Set these OUTSIDE this notebook (shell env vars, or a .env file loaded via
# python-dotenv and kept out of git) -- never hardcode credentials in a notebook
# that might get committed to a public repo.
#
# MLFLOW_TRACKING_URI      = https://dagshub.com/<your-username>/SellWise.mlflow
# MLFLOW_TRACKING_USERNAME = <your DagsHub username>
# MLFLOW_TRACKING_PASSWORD = <your DagsHub token, NOT your account password>

assert os.environ.get("MLFLOW_TRACKING_URI"), (
    "MLFLOW_TRACKING_URI not set. Set it (and MLFLOW_TRACKING_USERNAME/PASSWORD) "
    "as environment variables before running this notebook -- see comment above."
)

In [8]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    sales_train_path: Path
    sales_test_path: Path
    sell_prices_path: Path
    calendar_path: Path
    final_submission_path: Path
    metric_file_name: Path
    end_train: int
    p_horizon: int
    mlflow_uri: str

In [9]:
from SellWise.constants import *
from SellWise.utils.common import read_yaml, create_directories, save_json

In [10]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([(self.config.artifacts_root)])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        eval_config = self.config.model_evaluation
        ingestion_config = self.config.data_ingestion
        recursive_params = self.params.RECURSIVE_TRAINING

        create_directories([eval_config.root_dir])

        return ModelEvaluationConfig(
            root_dir=eval_config.root_dir,
            sales_train_path=ingestion_config.sales_path,
            sales_test_path=eval_config.sales_test_evaluation_path,
            sell_prices_path=ingestion_config.sell_prices_path,
            calendar_path=ingestion_config.calendar_path,
            final_submission_path=eval_config.final_submission_path,
            metric_file_name=eval_config.metric_file_name,
            end_train=recursive_params.end_train,
            p_horizon=recursive_params.p_horizon,
            mlflow_uri=os.environ["MLFLOW_TRACKING_URI"]
        )

In [11]:
import numpy as np
import pandas as pd
import mlflow
from SellWise import logger

In [13]:
# The 12 M5 aggregation levels, exactly matching the official
# Estimate weights.R / Estimate WRMSSE.R grouping keys.
LEVEL_KEYS = {
    "Level1":  [],
    "Level2":  ["state_id"],
    "Level3":  ["store_id"],
    "Level4":  ["cat_id"],
    "Level5":  ["dept_id"],
    "Level6":  ["state_id", "cat_id"],
    "Level7":  ["state_id", "dept_id"],
    "Level8":  ["store_id", "cat_id"],
    "Level9":  ["store_id", "dept_id"],
    "Level10": ["item_id"],
    "Level11": ["state_id", "item_id"],
    "Level12": ["item_id", "store_id"],
}


class ModelEvaluation:
    """
    Computes the true 12-level hierarchical WRMSSE, matching the official
    M5 organizers' Estimate weights.R / Estimate WRMSSE.R exactly:
    bottom-level (item x store) predictions get summed up to all 12
    aggregation levels, scored per level (weighted by each group's dollar
    sales, scaled by its own naive-forecast in-sample error), then the 12
    per-level scores are averaged into one final number.
    """

    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def load_data(self):
        logger.info("Loading train/test/prices/calendar/submission")
        self.sales_train = pd.read_csv(self.config.sales_train_path)
        self.sales_test = pd.read_csv(self.config.sales_test_path)
        self.sell_prices = pd.read_csv(self.config.sell_prices_path)
        self.calendar = pd.read_csv(self.config.calendar_path)
        self.submission = pd.read_csv(self.config.final_submission_path)

    def _build_frames(self):
        """Aligns train (meta + history), test (actuals), and forecast
        (our ensembled predictions) into matching row order, keyed by
        (item_id, store_id)."""
        meta_cols = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]
        train_day_cols = [c for c in self.sales_train.columns if c.startswith("d_")]

        meta = self.sales_train[meta_cols].reset_index(drop=True)
        train_wide = self.sales_train[train_day_cols].reset_index(drop=True)

        # sales_test_evaluation.csv has no 'id' column -- join on item_id+store_id
        test_wide = meta.merge(
            self.sales_test, on=meta_cols, how="left"
        )
        test_day_cols = [c for c in self.sales_test.columns if c.startswith("d_")]
        test_wide = test_wide[test_day_cols]

        # submission has 'id' (e.g. ITEM_1_001_CA_1_evaluation) + F1..F28 --
        # reconstruct item_id/store_id from sales_train's own id mapping to join.
        id_map = self.sales_train[["id"] + meta_cols]
        forecast_full = id_map.merge(self.submission, on="id", how="left")
        f_cols = [c for c in self.submission.columns if c.startswith("F")]
        forecast_wide = forecast_full[f_cols].reset_index(drop=True)
        forecast_wide.columns = test_day_cols  # align column labels for direct subtraction

        assert len(meta) == len(train_wide) == len(test_wide) == len(forecast_wide), (
            "Row count mismatch after alignment -- check for missing ids in submission "
            "or missing (item_id, store_id) pairs in sales_test_evaluation.csv"
        )
        return meta, train_wide, test_wide, forecast_wide

    def _compute_dollar_sales_weights(self, meta, train_wide):
        """Level12 weight basis: dollar sales in the LAST 28 days of the
        training window (matching Estimate weights.R exactly)."""
        train_day_cols = list(train_wide.columns)
        last_28_cols = train_day_cols[-28:]

        last_28_days = last_28_cols  # e.g. d_1914..d_1941
        day_to_week = self.calendar.set_index("d")["wm_yr_wk"].to_dict()

        long = train_wide[last_28_days].copy()
        long["item_id"] = meta["item_id"].values
        long["store_id"] = meta["store_id"].values
        long = long.melt(id_vars=["item_id", "store_id"], var_name="d", value_name="quantity")
        long["wm_yr_wk"] = long["d"].map(day_to_week)

        long = long.merge(
            self.sell_prices, on=["item_id", "store_id", "wm_yr_wk"], how="left"
        )
        long["sell_price"] = long["sell_price"].fillna(0)
        long["dollar"] = long["quantity"] * long["sell_price"]

        dollar_sales = long.groupby(["item_id", "store_id"])["dollar"].sum()
        # Reindex to match meta's row order exactly
        dollar_sales = dollar_sales.reindex(
            pd.MultiIndex.from_frame(meta[["item_id", "store_id"]])
        ).reset_index(drop=True).fillna(0)
        return dollar_sales

    @staticmethod
    def _aggregate_level(meta, train_wide, test_wide, forecast_wide, dollar_sales, keys):
        if not keys:
            return (
                train_wide.sum(axis=0).to_frame().T,
                test_wide.sum(axis=0).to_frame().T,
                forecast_wide.sum(axis=0).to_frame().T,
                pd.Series([dollar_sales.sum()]),
            )
        grp_idx = meta.groupby(keys).ngroup()
        return (
            train_wide.groupby(grp_idx).sum(),
            test_wide.groupby(grp_idx).sum(),
            forecast_wide.groupby(grp_idx).sum(),
            dollar_sales.groupby(grp_idx).sum(),
        )

    @staticmethod
    def _rmsse_row(insample, outsample, forecast):
        insample = np.asarray(insample, dtype=float)
        nz = np.nonzero(insample > 0)[0]
        start = nz[0] if len(nz) else 0
        trimmed = insample[start:]
        if len(trimmed) < 2:
            return np.nan
        scale = np.mean(np.diff(trimmed) ** 2)
        error = np.mean((np.asarray(forecast) - np.asarray(outsample)) ** 2)
        if scale == 0 or np.isnan(scale):
            return np.nan
        return np.sqrt(error / scale)

    def compute_wrmsse(self):
        meta, train_wide, test_wide, forecast_wide = self._build_frames()
        dollar_sales = self._compute_dollar_sales_weights(meta, train_wide)
        grand_total = dollar_sales.sum()

        level_scores = {}
        for level_name, keys in LEVEL_KEYS.items():
            grp_train, grp_test, grp_forecast, grp_weight = self._aggregate_level(
                meta, train_wide, test_wide, forecast_wide, dollar_sales, keys
            )
            weight_share = grp_weight / grand_total

            rmsse_vals = np.array([
                self._rmsse_row(
                    grp_train.iloc[i].values, grp_test.iloc[i].values, grp_forecast.iloc[i].values
                )
                for i in range(len(grp_train))
            ])
            contribs = rmsse_vals * weight_share.values
            level_scores[level_name] = float(np.nansum(contribs))
            logger.info(f"{level_name}: {len(grp_train)} groups, WRMSSE={level_scores[level_name]:.5f}")

        final_wrmsse = float(np.mean(list(level_scores.values())))
        level_scores["WRMSSE_final"] = final_wrmsse
        logger.info(f"Final WRMSSE (mean across 12 levels): {final_wrmsse:.5f}")
        return level_scores

    def log_into_mlflow(self):
        self.load_data()
        scores = self.compute_wrmsse()

        save_json(path=Path(self.config.metric_file_name), data=scores)

        mlflow.set_tracking_uri(self.config.mlflow_uri)
        with mlflow.start_run():
            mlflow.log_param("end_train", self.config.end_train)
            mlflow.log_param("p_horizon", self.config.p_horizon)
            for level_name, score in scores.items():
                mlflow.log_metric(level_name, score)

        logger.info(f"Metrics saved to {self.config.metric_file_name} and logged to MLflow")
        return scores


In [14]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    scores = model_evaluation.log_into_mlflow()
except Exception as e:
    raise e

[2026-09-26 16:50:27,282: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-09-26 16:50:27,315: INFO: common: yaml file: params.yaml loaded successfully]
[2026-09-26 16:50:27,331: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-09-26 16:50:27,333: INFO: common: created directory at: artifacts]
[2026-09-26 16:50:27,334: INFO: common: created directory at: artifacts/model_evaluation]
[2026-09-26 16:50:27,335: INFO: 200447592: Loading train/test/prices/calendar/submission]
[2026-09-26 16:50:34,394: INFO: 200447592: Level1: 1 groups, WRMSSE=1.24714]
[2026-09-26 16:50:34,844: INFO: 200447592: Level2: 3 groups, WRMSSE=1.18333]
[2026-09-26 16:50:35,285: INFO: 200447592: Level3: 10 groups, WRMSSE=1.14815]
[2026-09-26 16:50:35,759: INFO: 200447592: Level4: 3 groups, WRMSSE=1.27168]
[2026-09-26 16:50:36,217: INFO: 200447592: Level5: 7 groups, WRMSSE=1.51536]
[2026-09-26 16:50:36,693: INFO: 200447592: Level6: 9 groups, WRMSSE=1.21776]
[2026-09-26 16:50:37,112: